# 🎲 QI identification
Uso `seed_dict` creato manualmente, per andare a definire categorie ed eventuali gerarchie per applicare generalizzazioni successive

## 🔢 Install dependencies

In [ ]:
# !pip install -q sentence-transformers spacy numpy

## **`seed_dict`** - import **json**



In [ ]:
import json

with open("seed_dict.json", encoding="utf-8") as f:
    seed_dict = json.load(f)


In [ ]:
print(seed_dict.keys())

dict_keys(['education', 'location', 'nationality', 'occupation', 'political_role', 'religion', 'temporal'])


## 0-shot *QI identification*
Seleziona l'input text prima di avviare lo script

In [ ]:
# text = "The retired firefighter, a 67 year old widower originally from Sarajevo, was admitted to a rehabilitation clinic after suffering a stroke; his colleagues described him as a devout Baptist and a staunch pacifist."
# text = "The elderly architect, a 75 year old bachelor from Sarajevo, was treated for leukemia at the local medical center; he is a lifelong Lutheran and a dedicated gardener."
# text = "Walter Extra is a German award-winning aerobatic pilot, mechanical engineer, chief aircraft designer, and founder of Extra Flugzeugbau (Extra Aircraft Construction), a manufacturer of aerobatic aircraft. He began his flight training in gliders before transitioning to powered aircraft for aerobatics. After building and flying a Pitts Special, he developed and flew his own Extra EA-230. Extra started designing aircraft after competing in the 1982 World Aerobatic Championships. His aircraft designs revolutionized aerobatic flying and continue to dominate world competitions. German pilot Klaus Schrodt won a world championship flying an aircraft produced by Extra Flugzeugbau. Walter Extra has designed a range of high-performance aircraft, including unlimited aerobatic airplanes and turboprop transport aircraft."
text = "Lauren Danielle drain born December 31, 1985 is an American nurse and author known for writing the 2013 book banished, which chronicles her experiences and eventual banishment from the controversial Westboro Baptist church."

TAU = 0.55  # Threshold fisso
MARGIN = 0.02  # Distanza minima dalla seconda categoria

### Finding centroid of **each category**

In [ ]:
# in:  seed_dict
# out: model, category_names, centroid_matrix (7, 768)
import numpy as np
from sentence_transformers import SentenceTransformer

# all-mpnet-base-v2 : dimensione = 768
# all-MiniLM-L6-v2  : dimensione = 384
model = SentenceTransformer("all-mpnet-base-v2")

def centroids(model):
    cats = sorted(seed_dict)
    mat = np.stack(
        [model.encode([t.lower() for t in seed_dict[c]], normalize_embeddings=True).mean(0) for c in cats]
    )
    return cats, mat / np.linalg.norm(mat, axis=1, keepdims=True)

category_names, centroid_matrix = centroids(model)
print(centroid_matrix.shape, category_names)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(7, 768) ['education', 'location', 'nationality', 'occupation', 'political_role', 'religion', 'temporal']


In [ ]:
# @title Scarico modelli `NTLK`

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

True

### **`SpaCy`** - NER

In [ ]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 13.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import spacy

# Rimosso nltk.corpus.names siccome non era un filtro efficiente passo a NER
_nlp_ner = spacy.load("en_core_web_md", disable=["parser", "lemmatizer"])

# Ottengo person_token con ent.label_ = PERSON
def get_person_entities(raw_text: str) -> tuple[list[tuple[int, int]], list[str]]:
    ents = [e for e in _nlp_ner(raw_text).ents if e.label_ == "PERSON"]
    return [(e.start_char, e.end_char) for e in ents], list(
        dict.fromkeys(e.text for e in ents)
    )

_person_spans, _persons = get_person_entities(text)
print(_persons)

['Lauren Danielle']


### Tokenize **`Sentence`**

In [ ]:
from nltk import pos_tag
from nltk.tokenize import TreebankWordTokenizer
from nltk.tokenize.punkt import PunktTokenizer

# Uso TreebankWordTokenizer (https://www.nltk.org/api/nltk.tokenize.TreebankWordTokenizer.html)
_TOK = TreebankWordTokenizer()

# Uso PunktTokenizer (https://www.nltk.org/api/nltk.tokenize.punkt.html)
_SENT = PunktTokenizer("english")

def _token_spans(txt: str) -> list[tuple[int, int]]:
    return [
        (s + a, s + b)
        for s, e in _SENT.span_tokenize(txt)
        for a, b in _TOK.span_tokenize(txt[s:e])
    ]

def segments(txt: str):
    groups, cur = [], []
    for (w, tag), (a, b) in zip(pos_tag([txt[a:b] for a, b in _token_spans(txt)]), _token_spans(txt)):
        if any(c.isalnum() for c in w):
            cur.append((w, tag, a, b))
        elif cur:
            groups.append(cur)
            cur = []
    if cur:
        groups.append(cur)
    return groups

tagged_groups = segments(text)

#### **`NTLK`** variable and functions for *POS_TAG*

In [ ]:
from itertools import product

NOUN = ("NN", "NNS", "NNP", "NNPS")

# POS tag da considerare validi
_SHAPES = [
    ("JJ", "N"),
    ("N", "N"),
    ("N", "CD"),  # January 1953
    ("CD", "JJ"),
    ("CD", "N"),  # 1982 championships
    ("CD", "N", "CD"),  # 23 July 1846, 19 January 1953
    ("CD", "N", "JJ"),  # 67 year old
    ("JJ", "JJ", "N"),
    ("JJ", "N", "N"),
    ("N", "N", "N"),
    ("N", "IN", "N"),  # University of Oxford
]

VALID_POS = {
    p for shape in _SHAPES
    for p in product(*[NOUN if t == "N" else (t,) for t in shape])
}

UNIGRAM_OK = {"NN", "NNS", "NNP", "NNPS", "CD", "JJ"}
BAD_MIDDLE = {"CC", "DT", "WP", "WRB", "MD", "VBD", "VBZ", "VBP", "VBN", "VBG", "TO", "RP"}

def is_valid_ngram(pos: tuple[str, ...]) -> bool:
    if len(pos) == 1:
        return pos[0] in UNIGRAM_OK
    if any(t in BAD_MIDDLE for t in pos[1:-1]):
        return False
    return pos in VALID_POS

#### Extracting **`n-gram`** from *token*

In [ ]:
def _keep(ng: str, n: int) -> bool:
    if n > 1:
        return True
    return len(ng) >= 3 and (not ng.isdigit() or len(ng) == 4)


# Includo anche 1-gram per individuare termini ('nurse', 'baptist', ...)
def build_ngrams(txt: str, person_spans: list[tuple[int, int]]) -> dict:
    out = {}
    for group in segments(txt):
        for n in (1, 2, 3):
            for i in range(len(group) - n + 1):
                window = group[i : i + n]
                if not is_valid_ngram(tuple(t for _, t, _, _ in window)):
                    continue
                start, end = window[0][2], window[-1][3]
                # scarta se interseca un PERSON
                if any(start < pe and ps < end for ps, pe in person_spans):
                    continue
                phrase = txt[start:end]
                if not _keep(phrase, n):
                    continue
                out[(start, end)] = phrase
    return out

filtered_ngrams = build_ngrams(text, _person_spans)

#### Evaluating **`CosineSimilarity`**

In [ ]:
_emb_cache: dict[str, np.ndarray] = {}

def encode(phrases: list[str]) -> np.ndarray:
    new = [p for p in dict.fromkeys(phrases) if p not in _emb_cache]
    if new:
        _emb_cache.update(zip(new, model.encode(new, normalize_embeddings=True, batch_size=256)))
    return np.stack([_emb_cache[p] for p in phrases])

def classify(ngrams: dict) -> list[dict]:
    if not ngrams:
        return []

    keys = list(ngrams)
    sim = encode([ngrams[k].lower() for k in keys]) @ centroid_matrix.T
    best_idx = sim.argmax(1)
    best_sim = sim.max(1)

    sorted_sim = np.sort(sim, axis=1)
    margin = sorted_sim[:, -1] - sorted_sim[:, -2] if sim.shape[1] > 1 else np.full(len(keys), np.inf)

    return [
        {
            "start": start,
            "end": end,
            "category": category_names[best_idx[i]],
            "score": float(best_sim[i]),
            "surface": ngrams[(start, end)],
            "source": "cosine",
        }
        for i, (start, end) in enumerate(keys)
        if margin[i] >= MARGIN
    ]

detected = classify(filtered_ngrams)

# aggiunge i PERSON dalla NER
person_spans = [
    {
        "start": a, "end": b,
        "category": "person",
        "score": 1.0,
        "surface": text[a:b],
        "source": "ner",
    }
    for a, b in _person_spans
]

results = person_spans + [s for s in detected if s["score"] >= TAU]

### Output

In [ ]:
print("Testo:\n" + text)

print("\nDetected PII:\n" + "-" * 55)
for s in sorted(results, key=lambda s: -s["score"]):
    print(f'[{s["category"].upper()}] {s["start"]:>4}-{s["end"]:<4} {s["surface"]!r} ({s["score"]:.2%})')

Testo:
Lauren Danielle drain born December 31, 1985 is an American nurse and author known for writing the 2013 book banished, which chronicles her experiences and eventual banishment from the controversial Westboro Baptist church.

Detected PII:
-------------------------------------------------------
[PERSON]    0-15   'Lauren Danielle' (100.00%)
[NATIONALITY]   51-59   'American' (82.28%)
[RELIGION]  208-215  'Baptist' (81.36%)
[RELIGION]  208-222  'Baptist church' (76.51%)
[TEMPORAL]   27-35   'December' (73.63%)
[RELIGION]  216-222  'church' (69.30%)
[TEMPORAL]   99-103  '2013' (63.13%)
[RELIGION]  199-215  'Westboro Baptist' (62.67%)
[RELIGION]  199-222  'Westboro Baptist church' (62.34%)
[TEMPORAL]   27-38   'December 31' (60.75%)
[TEMPORAL]   40-44   '1985' (58.79%)
[OCCUPATION]   70-76   'author' (58.35%)
[OCCUPATION]   60-65   'nurse' (56.63%)


In [ ]:
def apply_spans(text: str, spans: list[dict]) -> tuple[str, list[dict]]:
    kept = []

    for span in sorted(spans, key=lambda s: (-(s["end"] - s["start"]), -s["score"])):
        if not any(span["end"] > o["start"] and span["start"] < o["end"] for o in kept):
            kept.append(span)
    kept.sort(key=lambda s: s["start"])

    out, prev = [], 0
    for span in kept:
        label = span["category"]
        out.append(text[prev:span["start"]])
        out.append(f"[{label.upper()}]")
        prev = span["end"]
    out.append(text[prev:])
    return "".join(out), kept

text_anon, kept = apply_spans(text, results)
print("\nTesto anonimizzato:\n" + "-" * 55)
print(text_anon)


Testo anonimizzato:
-------------------------------------------------------
[PERSON] drain born [TEMPORAL], [TEMPORAL] is an [NATIONALITY] [OCCUPATION] and [OCCUPATION] known for writing the [TEMPORAL] book banished, which chronicles her experiences and eventual banishment from the controversial [RELIGION].


### Creo `corpus.cosine.jsonl`

In [ ]:
IN_PATH  = "data/corpus.jsonl"
OUT_TAUS = [0.40, 0.50, 0.55, 0.60]

_RUN_CORPUS = False

In [ ]:
from datetime import datetime, timezone

from tqdm import tqdm
import os

meta = {
    "st_model": "all-mpnet-base-v2",
    "spacy_model": "en_core_web_md",
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

if _RUN_CORPUS:

  os.makedirs("results", exist_ok=True)

  fouts = {
      t: open(f"results/corpus.cosine.t{int(t * 100)}.jsonl", "w", encoding="utf-8")
      for t in OUT_TAUS
  }
  n = 0

  with open(IN_PATH, encoding="utf-8") as fin:
      for line in tqdm(fin, desc="Cosine", unit="doc"):
          if not line.strip():
              continue
          rec = json.loads(line)
          doc = rec["text"]
          spans, _ = get_person_entities(doc)
          found = classify(build_ngrams(doc, spans))
          persons = [
              {
                  "start": a,
                  "end": b,
                  "category": "person",
                  "score": 1.0,
                  "surface": doc[a:b],
                  "source": "ner",
              }
              for a, b in spans
          ]

          for tau, fout in fouts.items():
              anon, kept = apply_spans(
                  doc, persons + [s for s in found if s["score"] >= tau]
              )
              fout.write(
                  json.dumps(
                      {
                          "id": rec["id"],
                          "title": rec.get("title"),
                          "url": rec.get("url"),
                          "box": rec["box"],
                          "text": doc,
                          "text_anon": anon,
                          "entities": [s for s in kept if s["category"] != "person"],
                          "method": {0.40: "M3", 0.50: "M4", 0.55: "M5"}.get(tau, f"cosine_t{int(tau * 100)}"),
                          "meta": {"tau": tau, **meta},
                      },
                      ensure_ascii=False,
                  )
                  + "\n"
              )
          n += 1

  for f in fouts.values():
      f.close()
  print(f"{n:,} documenti -> {len(OUT_TAUS)} file")

  secrets = json.load(open("canary_secrets.json", encoding="utf-8"))
  for tau in OUT_TAUS:
      cur = {
          r["id"]: r
          for r in (
              json.loads(l)
              for l in open(f"results/corpus.cosine.t{int(tau * 100)}.jsonl", encoding="utf-8")
          )
      }
      persi = sum(
          v["secret"] not in cur[c]["text_anon"] for c, v in secrets.items() if c in cur
      )
      ents = sum(len(r["entities"]) for r in cur.values()) / len(cur)
      print(
          f"tau={tau:.2f}  {ents:5.1f} entita'/doc (M1: 9.8)  secret rimossi {persi}/{len(secrets)}"
      )